In [36]:
import optuna
from new_model_class import Coating_train_2
import lightning as L
from pytorch_lightning import Trainer
import pandas as pd
from pytorch_lightning.loggers import CSVLogger
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
import torch
from torch.utils.data import TensorDataset, DataLoader
from pytorch_lightning.callbacks import EarlyStopping
import numpy as np
from datetime import datetime
import os

In [8]:
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

file_name = "rawdata-2026.01.21-NaN_to_0.csv"
dataset_path = os.path.join(current_dir, "rawdata", file_name)

print(f"Dataset path: {dataset_path}")

Dataset path: /Users/zydeng/Library/CloudStorage/OneDrive-宁波东方理工大学（暂名）/EIT Academic/proj-2023.06.30-AI+powder/Colour+AI/proj-2025.01.21-try/rawdata/rawdata-2026.01.21-NaN_to_0.csv


In [9]:
# 获取当前日期和时间，并格式化为字符串
current_datetime = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# 定义文件名，并将日期和时间添加到文件名中
filename = f"tuning_results-{current_datetime}.csv"
logname = f"logs-{current_datetime}"

# 创建日志目录
log_dir = os.path.join(os.getcwd(), logname)
os.makedirs(log_dir, exist_ok=True)

In [29]:
label_features = ["gloss_value", "L", "A", "B"]
drop_features = label_features

rawdata = pd.read_csv(dataset_path)
features = rawdata.drop(drop_features, axis=1)
labels = rawdata[label_features].drop("gloss_value", axis=1)

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_features = scaler.fit_transform(features)
scaled_labels = scaler.fit_transform(labels)

scaled_features, scaled_labels = features.to_numpy(), labels.to_numpy()

features_tensor = torch.tensor(scaled_features, dtype=torch.float32)
labels_tensor = torch.tensor(scaled_labels, dtype=torch.float32)
dataset = TensorDataset(features_tensor, labels_tensor)

print("\n------------------------------------------------------")
print("\nThe length of features is ", features_tensor.shape[1])
print("\nThe length of labels is ", labels_tensor.shape[1])
print("\n------------------------------------------------------")


------------------------------------------------------

The length of features is  461

The length of labels is  3

------------------------------------------------------


In [39]:
kf = KFold(n_splits=5, shuffle=True)
val_losses = []
r2_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    input_size = features_tensor.shape[1]
    output_size = labels_tensor.shape[1]

    learning_rate = 1e-3
    n_layers = 4
    neurons_per_layer = [32] * n_layers
    dropout_rate = 0.1
    activation_fn = "gelu"
    batch_size = 64
    delta = 1.0

    model = Coating_train_2(
        input_size,
        output_size,
        n_layers,
        neurons_per_layer,
        learning_rate,
        dropout_rate,
        activation_fn,
        delta,
    )

    train_subset = torch.utils.data.Subset(dataset, train_idx)
    val_subset = torch.utils.data.Subset(dataset, val_idx)

    train_loader = DataLoader(
        train_subset, batch_size=batch_size, shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        val_subset, batch_size=batch_size, shuffle=False, num_workers=0
    )

    test_loader = val_loader

    # 判断是否有可用的GPU
    if torch.backends.mps.is_available():
        accelerator = "mps"
    elif torch.cuda.is_available():
        accelerator = "gpu"
    else:
        accelerator = "cpu"

    # 定义早停回调函数
    early_stop_callback = EarlyStopping(
        monitor="loss_val", patience=50, verbose=False, mode="min"
    )

    # 设置CSVLog日志记录
    logger = CSVLogger(
        save_dir=log_dir,
        name=f"trial_0_fold_{fold}",
    )

    trainer = Trainer(
        max_epochs=1000,
        accelerator=accelerator,
        callbacks=[early_stop_callback],
        default_root_dir=log_dir,
        log_every_n_steps=20,
        logger=logger,
    )

    trainer.fit(model, train_loader, val_loader)

    # final_val_loss = trainer.callback_metrics["loss_val"]
    # val_losses.append(final_val_loss.item())
    val_loss_best = trainer.early_stopping_callback.best_score.item()
    val_losses.append(val_loss_best)

    test_result = trainer.test(model, dataloaders=test_loader, verbose=False)
    r2_scores.append(test_result[0]["R2_score"])

# min_val_loss = np.min(val_losses)
# max_r2_score = np.max(r2_scores)
mid_val_loss = np.median(val_losses)
mid_r2_score = np.median(r2_scores)

print("\n------------------------------------------------------")
print(f"Trial {trial.number}")
print(f"Median val loss: {mid_val_loss}")
print(f"Median R2 score: {mid_r2_score}")
print("------------------------------------------------------")

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name            | Type       | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | network         | Sequential | 18.1 K | train | 0    
1 | criterion       | HuberLoss  | 0      | train | 0    
2 | r2_score_metric | R2Score    | 0      | train | 0    
---------------------------------------------------------------
18.1 K    Trainable params
0         Non-trainable params
18.1 K    Total params
0.072     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
0         Total Flops


/opt/homebrew/Caskroom/miniconda/base/envs/pyday/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 159:  75%|███████▌  | 177/236 [00:01<00:00, 116.64it/s, v_num=1, loss_train_step=1.820, loss_val=2.970, loss_train_epoch=2.480]


Detected KeyboardInterrupt, attempting graceful shutdown ...


Epoch 159:  75%|███████▌  | 178/236 [00:01<00:00, 116.66it/s, v_num=1, loss_train_step=1.820, loss_val=2.970, loss_train_epoch=2.480]

SystemExit: 1

/opt/homebrew/Caskroom/miniconda/base/envs/pyday/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
